# Access to DP2

This notebook is the **entry point** to LSST Data Preview 2 (DP2).  
It demonstrates how to access processed data products stored in the  
USDF butler repository, map observations to sky tracts/patches, and  
visualise visit statistics per band and per Deep Drilling Field (DDF).


- **Author:** Sylvie Dagoret-Campagne  
- **Creation date:** 2026-03-07  
- **Environment:** USDF RSP — `LSST` kernel (`lsst_distrib` stack)


## 1. Imports

Standard scientific stack plus the LSST middleware:  
- `lsst.daf.butler` — data access layer  
- `lsst.afw.display` — image display utilities  
- `lsst.geom` — sky coordinate geometry  
- `lsst.skymap` — tract/patch sky tessellation


In [ ]:
import lsst.pipe.base

print(lsst.pipe.base.__version__)

In [ ]:
import sys
import matplotlib.pyplot as plt
import lsst.afw.display as afwDisplay
from lsst.geom import SpherePoint, degrees
from lsst.afw.image import ExposureF
from lsst.skymap import PatchInfo, Index2D
import numpy as np
import pandas as pd
from astropy.time import Time
# %matplotlib widget
import seaborn as sns

In [ ]:
# LSST data access middleware
from lsst.daf.butler import Butler

In [ ]:
def get_uris_from_butler(butler, datasetType, collections=None, **query_kwargs):
    """
    Return a list of URIs for a given datasetType in a Butler collection.

    Parameters
    ----------
    butler : lsst.daf.butler.Butler
        The Butler object
    datasetType : str
        The dataset type, e.g. 'raw', 'calexp'
    collections : list of str, optional
        Butler collections to query
    query_kwargs : dict
        Additional query parameters, e.g. visit=123, filter='r'

    Returns
    -------
    uris : list of str
        File paths of the datasets
    """
    refs = butler.registry.queryDatasets(datasetType, collections=collections, **query_kwargs)
    return [butler.getURI(ref) for ref in refs]

In [ ]:
def get_refs_from_butler(
    butler,
    datasetType,
    collections=None,
    return_uris=False,
    **query_kwargs
):
    """
    Get dataset references (or URIs) from a Butler, safely handling numpy types.

    Parameters
    ----------
    butler : lsst.daf.butler.Butler
        Butler instance.
    datasetType : str
        Dataset type to query (e.g., "calexp", "raw", "src").
    collections : str or list of str, optional
        Butler collections to query.
    return_uris : bool, default False
        If True, return URIs instead of refs.
    **query_kwargs :
        Query parameters, e.g., visit=[...], ccd=[...], filter=['r','g'].

    Returns
    -------
    list of DatasetRef or list of str
        List of references or URIs.
    """

    # Convert numpy types to Python native types to avoid Butler query errors
    safe_kwargs = {}
    for k, v in query_kwargs.items():
        if isinstance(v, np.ndarray):
            safe_kwargs[k] = [int(x) if np.issubdtype(type(x), np.integer) else x for x in v]
        elif isinstance(v, (np.integer, np.int64, np.int32)):
            safe_kwargs[k] = int(v)
        else:
            safe_kwargs[k] = v

    # Query dataset references
    refs = butler.registry.queryDatasets(
        datasetType,
        collections=collections,
        **safe_kwargs
    )

    if return_uris:
        return [butler.getURI(ref) for ref in refs]

    return refs

## 2. Configuration

Select the Butler repository, the DRP processing collection, the sky map name,  
and the instrument.  Multiple DRP collections are listed for reference:  
uncomment the desired one before running.


- See collection here : https://usdf-rsp.slac.stanford.edu/plot-navigator
- See Campaign : https://rubinobs.atlassian.net/wiki/spaces/DM/pages/661192727/LSSTCam+Intermittent+DRP+Runs

In [ ]:
# ── Butler repository and DRP collection ──────────────────────────────────────
# Available DRP collections (uncomment the desired one):
#   'LSSTCam/runs/DRP/v30_0_1/DM-54061'
#   'LSSTCam/runs/DRP/v30_0_4/DM-54249'   <- currently selected
#   'LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881'
REPO = 'dp2_prep'
collection =  'LSSTCam/runs/DRP/v30_0_4/DM-54249'
#COLLECTION  = "LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545"
skymapName = "lsst_cells_v2"
instrument = "LSSTCam"

# ── Date range for exposure queries ───────────────────────────────────────────
date_start = 20260101      # YYYYMMDD — earliest day_obs to include
date_selection = 20260310  # YYYYMMDD — reference date for the analysis

# ── Build WHERE clauses for Butler registry queries ───────────────────────────
where_clause = "instrument = '" + f"{instrument}" + "'"
where_clause_date = where_clause + f"and day_obs >= {date_start}"

# ── Instantiate the Butler and its registry ───────────────────────────────────
butler = Butler(REPO, collections=collection)
registry = butler.registry

In [ ]:
# Retrieve the sky tessellation (skyMap) from the butler.
# The 'lsst_cells_v2' skymap divides the full sky into tracts and patches.
# Error handling is included because the skymap may not be present in all collections.
try:
    skymap = butler.get("skyMap", skymap=skymapName, collections=collection)
except Exception as inst:
    print(type(inst))   # exception type
    print(inst.args)    # arguments stored in .args
    print(inst)         # full message via __str__

In [ ]:
# Inspect the skymap object (number of tracts, pixel scale, etc.)
skymap

In [ ]:
# ── Define an empty exposure DataFrame with explicit dtypes ──────────────────
# This pre-declaration ensures that column types are correct even if no rows
# are appended (e.g., when the query returns no results).
df_exposure = pd.DataFrame(
    {
        "id": pd.Series(dtype="int"),
        "obs_id": pd.Series(dtype="int"),
        "day_obs": pd.Series(dtype="int"),
        "seq_num": pd.Series(dtype="int"),
        "time_start": pd.Series(dtype="str"),
        "time_end": pd.Series(dtype="str"),
        "type": pd.Series(dtype="str"),
        "target": pd.Series(dtype="str"),
        "filter": pd.Series(dtype="str"),
        "zenith_angle": pd.Series(dtype="float"),
        "expos": pd.Series(dtype="float"),
        "ra": pd.Series(dtype="float"),
        "dec": pd.Series(dtype="float"),
        "skyangle": pd.Series(dtype="float"),
        "azimuth": pd.Series(dtype="float"),
        "zenith": pd.Series(dtype="float"),
        "science_program": pd.Series(dtype="str"),
        "jd": pd.Series(dtype="float"),
        "mjd": pd.Series(dtype="float"),
    }
)

## 3. Retrieve exposure dimension records

Query the Butler registry for all `exposure` dimension records matching  
the selected instrument and date range.  
For each record the Julian Date (JD) is converted to MJD and ISO-8601 (ISOT)  
using `astropy.time.Time`, then all relevant metadata are stored in a list  
of dictionaries before being assembled into a `pandas.DataFrame`.

> **Note:** The loop is capped at 50 000 rows (`count > 50000`) to prevent  
> runaway execution on very large collections.


In [ ]:
# Accumulate exposure metadata in a list of dicts before building the DataFrame
rows = []
for count, info in enumerate(registry.queryDimensionRecords("exposure", where=where_clause_date)):
    try:
        # Convert Butler TAI JD timestamps to MJD and ISOT using astropy.time
        jd_start = info.timespan.begin.value
        jd_end   = info.timespan.end.value
        t_start  = Time(jd_start, format="jd", scale="utc")
        t_end    = Time(jd_end,   format="jd", scale="utc")

        if count == 0:
            print("===== Time Conversion Debug Info (first row) =====")
            print(f"JD start  : {jd_start}  |  MJD start : {t_start.mjd}")
            print(f"ISOT start: {t_start.isot}")
            print("==================================================")

        row = {
            "id":               info.id,
            "obs_id":           info.obs_id,
            "day_obs":          info.day_obs,
            "seq_num":          info.seq_num,
            "time_start":       t_start.isot,
            "time_end":         t_end.isot,
            "type":             info.observation_type,
            "target":           info.target_name,
            "filter":           info.physical_filter,
            "zenith_angle":     info.zenith_angle,
            "expos":            info.exposure_time,
            "ra":               info.tracking_ra,
            "dec":              info.tracking_dec,
            "skyangle":         info.sky_angle,
            "azimuth":          info.azimuth,
            "zenith":           info.zenith_angle,
            "science_program":  info.science_program,
            "jd":               float(jd_start),
            "mjd":              float(t_start.mjd),
        }
        rows.append(row)
        if count > 50000:
            break

    except ValueError as e:
        print(f"ValueError at row {count}: {e}")
    except FileNotFoundError as e:
        print(f"FileNotFoundError at row {count}: {e}")
    except Exception as e:
        print(f"Unexpected error at row {count}: {type(e).__name__} — {e}")
        import traceback; traceback.print_exc()

In [ ]:
# Assemble the list of row dicts into a pandas DataFrame
df_exposure = pd.DataFrame(rows)
print(f"Total exposures retrieved: {len(df_exposure)}")

## 4. Filter science exposures

Retain only rows where `observation_type == 'science'` and reset the index.  
Engineering, bias, flat, and dark frames are discarded.


In [ ]:
df_science = df_exposure[df_exposure.type == "science"]
df_science.reset_index(drop=True, inplace=True)
print(f"Science exposures: {len(df_science)}")

In [ ]:
# Unique target names — useful to identify DDFs, WFD fields, etc.
df_science.target.unique()

## 5. Map exposures to sky tracts and patches

For each science exposure the pointing coordinates `(ra, dec)` are converted  
to a `lsst.geom.SpherePoint` and passed to `skymap.findTract` /  
`tract_info.findPatch` to retrieve the tract ID and the sequential patch index.  
The result is stored in new columns: `tract`, `patch`, and `patch_str`.


In [ ]:
def get_tract_patch(row, skymap):
    """Return the tract ID, sequential patch index, and 2-D patch index string
    for the sky position (row['ra'], row['dec']) in degrees."""
    if pd.isna(row['ra']) or pd.isna(row['dec']):
        return pd.Series({"tract": None, "patch": None, "patch_str": None})

    target_point = SpherePoint(row['ra'], row['dec'], degrees)
    tract_info   = skymap.findTract(target_point)
    patch_info   = tract_info.findPatch(target_point)

    return pd.Series({
        "tract":     tract_info.getId(),
        "patch":     patch_info.getSequentialIndex(),
        "patch_str": f"{patch_info.getIndex()[0]},{patch_info.getIndex()[1]}",
    })

In [ ]:
# Work on a copy to preserve df_science
df = df_science.copy()
# Extract the broad band name (e.g. 'r' from 'r_03') by splitting on '_'
df["band"] = df["filter"].apply(lambda x: x.split("_")[0])

In [ ]:
# Apply the tract/patch mapping row-by-row (may take a few minutes for large datasets)
df[['tract', 'patch', "patch_str"]] = df.apply(get_tract_patch, axis=1, args=(skymap,))

In [ ]:
df

In [ ]:
# List all unique target names present in the science exposures
print(list(df.target.unique()))

## 6. Focus on Deep Drilling Fields

Select subsets of exposures pointing at specific DDFs:  
- **COSMOS** — one of the most-observed extragalactic fields  
- **ELAIS-S1** — European Large-Area ISO Survey South 1 field


In [ ]:
# The target name can appear in different formats depending on the scheduler version
df_cosmos  = df[(df.target == 'ddf_cosmos, lowdust') | (df.target == '9813_lowdust, ddf_cosmos')]
df_elaiss1 = df[df.target == 'ddf_elaiss1, lowdust']

In [ ]:
df_cosmos

In [ ]:
df_elaiss1

## 7. Visit counts per tract and band — bar charts

Group all science exposures by `(tract, target)` tag and by band,  
then visualise the distribution as stacked bar charts.  
Both a vertical overview (all tracts) and a horizontal zoom on the  
selected DDF tracts are produced.


In [ ]:
# Sort by tract number so that bar chart labels are in a meaningful order
df = df.sort_values("tract")
# Create a composite tag: tract_number + '_' + target_name
df["tag"] = df["tract"].astype(str) + "_" + df["target"]
# Count visits per (tag, band) pair
grouped_tag = df.groupby(['tag', 'band']).size().reset_index(name='count')
# Keep the tag order as encountered in the sorted DataFrame
tag_order = df["tag"].drop_duplicates().tolist()

In [ ]:
# Canonical band order and plot colour mapping
band_order = ['u', 'g', 'r', 'i', 'z', 'y']
color_map = {
    'u': '#9b59b6',   # purple
    'g': '#2ecc71',   # green
    'r': '#e74c3c',   # red
    'i': '#e67e22',   # orange
    'z': '#3498db',   # blue
    'y': '#795548',   # brown
}

In [ ]:
grouped_tag

In [ ]:
List_Of_fields = list(grouped_tag["tag"].unique())

In [ ]:
# Print only the tags that correspond to Deep Drilling Fields
for field in List_Of_fields:
    if "ddf" in field:
        print(field)

In [ ]:
# Pivot to wide format: rows = tags, columns = bands
df_wide = grouped_tag.pivot(index='tag', columns='band', values='count').fillna(0)

# Keep only griz y for the overview (u is rare in early data)
band_order = ['g', 'r', 'i', 'z', 'y']
df_wide = df_wide[[b for b in band_order if b in df_wide.columns]]

In [ ]:
# ── Vertical stacked bar chart — all tracts ───────────────────────────────────
plt.figure(figsize=(20, 8))
bottom = pd.Series([0] * len(df_wide), index=df_wide.index)

for band in band_order:
    if band not in df_wide.columns:
        continue
    plt.bar(df_wide.index, df_wide[band], bottom=bottom,
            color=color_map[band], label=band, width=1)
    bottom += df_wide[band]

plt.xlabel("Visited fields / tract", fontsize=14)
plt.ylabel("Number of visits per band", fontsize=14)
plt.title("Stacked visits per band in each tract", fontsize=20, fontweight="bold")
plt.xticks(rotation=45, ha='right')
plt.legend(title="Band")
plt.tight_layout()
plt.show()

In [ ]:
# ── Horizontal stacked bar chart — all tracts (tall figure) ───────────────────
plt.figure(figsize=(16, 80))   # height scales with number of tracts
left = pd.Series([0] * len(df_wide), index=df_wide.index)

for band in band_order:
    if band not in df_wide.columns:
        continue
    plt.barh(y=df_wide.index, width=df_wide[band], left=left,
             color=color_map[band], label=band, height=1.0)
    left += df_wide[band]

plt.ylabel("Visited fields / tract", fontsize=14)
plt.xlabel("Number of visits per band", fontsize=14)
plt.title("Stacked visits per band in each tract (horizontal)", fontsize=20, fontweight="bold")
plt.legend(title="Band", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.ylim(-0.2, len(df_wide) - 0.2)
plt.tight_layout()
plt.show()

In [ ]:
# ── Manual list of DDF-related tract tags ─────────────────────────────────────
# These correspond to the six LSST DDFs and their adjacent tracts.
selected_tags = [
    "2233_ddf_edfs_a, lowdust",
    "2234_ddf_edfs_a, lowdust",
    "2234_lowdust, ddf_edfs_a",
    "2394_ddf_edfs_a, lowdust",
    "2394_lowdust, ddf_edfs_a",
    "2395_ddf_edfs_b, lowdust",
    "2396_ddf_edfs_b, lowdust",
    "2561_ddf_edfs_b, lowdust",
    "2876_ddf_elaiss1, lowdust",
    "2877_ddf_elaiss1, lowdust",
    "4848_ddf_ecdfs, lowdust",
    "4848_lowdust, ddf_ecdfs",
    "4849_lowdust, ddf_ecdfs",
    "5063_ddf_ecdfs, lowdust",
    "5063_lowdust, ddf_ecdfs",
    "8524_ddf_xmm_lss, lowdust",
    "8524_lowdust, ddf_xmm_lss",
    "8766_lowdust, ddf_xmm_lss",
    "9813_ddf_cosmos, lowdust",
    "9813_lowdust, ddf_cosmos"
]

In [ ]:
# Filter the grouped table to only the selected DDF tags
df_subset = grouped_tag[grouped_tag['tag'].isin(selected_tags)]

In [ ]:
df_wide = df_subset.pivot(index='tag', columns='band', values='count').fillna(0)
df_wide = df_wide[[b for b in band_order if b in df_wide.columns]]  # keep canonical band order

In [ ]:
# ── Horizontal stacked bar chart — DDF tracts only ────────────────────────────
plt.figure(figsize=(12, len(df_wide) * 0.5))  # height proportional to number of tags
left = pd.Series([0] * len(df_wide), index=df_wide.index)

for band in band_order:
    if band not in df_wide.columns:
        continue
    plt.barh(y=df_wide.index, width=df_wide[band], left=left,
             color=color_map[band], label=band, height=0.6)
    left += df_wide[band]

plt.xlabel("Number of visits per band")
plt.ylabel("Tag / Tract")
plt.title("Stacked visits per band (DDF tracts)", fontsize=18, fontweight="bold")
plt.legend(title="Band", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.ylim(-0.5, len(df_wide) - 0.5)
plt.tight_layout()
plt.show()

## 8. Available dataset types in the collection

List all dataset types present in the selected collection,  
filtering out pipeline-internal products (configs, logs, metrics, plots)  
to focus on science data products (calexp, src, dia_source, dia_object, …).


In [ ]:
FLAG_DUMP_DATASETS = True
if FLAG_DUMP_DATASETS:
    for datasetType in registry.queryDatasetTypes():
        if registry.queryDatasets(datasetType, collections=collection).any(
            execute=False, exact=False
        ):
            # Skip pipeline bookkeeping products
            if (
                ("_config"         not in datasetType.name)
                and ("_log"        not in datasetType.name)
                and ("_metadata"   not in datasetType.name)
                and ("_resource_usage" not in datasetType.name)
                and ("Plot"        not in datasetType.name)
                and ("Metric"      not in datasetType.name)
                and ("metric"      not in datasetType.name)
            ):
                print(datasetType)

## 9. Query raw datasets for ELAIS-S1 visits

Retrieve the Butler `DatasetRef` objects for all `raw` exposures  
belonging to the ELAIS-S1 DDF visits.  
The visit IDs are extracted from `df_elaiss1` and formatted into a  
Butler `WHERE` clause string.


In [ ]:
# Extract ELAIS-S1 visit IDs and cast to plain Python ints
visits_elais = df_elaiss1["id"].values
visit_list   = [int(v) for v in visits_elais]

In [ ]:
# Build the Butler WHERE clause and query raw datasets
visit_str = ",".join(str(v) for v in visit_list)
where = f"instrument='{instrument}' AND visit IN ({visit_str})"
refs = list(
    butler.registry.queryDatasets(
        "raw",
        collections=collection,
        where=where
    )
)

In [ ]:
# Number of raw CCD exposures found for ELAIS-S1 visits
len(refs)

In [ ]:
# Inspect the required dimensions for dia_source (needed for difference imaging queries)
print(butler.registry.getDatasetType("dia_source").dimensions)

In [ ]:
# Inspect the required dimensions for dia_object
print(butler.registry.getDatasetType("dia_object").dimensions)